# 06 · Statistical tests

Paired comparisons on the same claim ids:

- **main:** A2 (RAG + numeric verification) vs A3 (contradiction-aware)
- **second-look control:** SR (self-reflection) vs A3

McNemar's test on per-claim correctness and paired bootstrap confidence intervals for the accuracy and macro-F1 differences. Only claims with a finalized gold label and a prediction from both approaches count.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
RUN_ID = None
EXPERIMENT_ID = "claimverif-main-v1"
N_BOOT = 5000
SEED = 20261102

In [ ]:
from engine.stats import compare
from engine.notebook import banner, pick_run, describe_run, load_gold_labels, save_table
run = pick_run(EXPERIMENT_ID, RUN_ID)
assert run is not None, "no run found -- execute notebook 03 first"
describe_run(run)
gold = load_gold_labels()
comparisons = run["manifest"].get("comparisons") or [{"name": "main", "a": "A2_rag_numeric", "b": "A3_contradiction_aware"}]

def report(preds, gold, comparisons, n_boot=N_BOOT, seed=SEED):
    rows = []
    for c in comparisons:
        pa = [p for p in preds if p["approach"] == c["a"]]
        pb = [p for p in preds if p["approach"] == c["b"]]
        r = compare(pa, pb, gold, n_boot=n_boot, seed=seed)
        if not r["n"]:
            rows.append({"comparison": c["name"], "a": c["a"], "b": c["b"], "n": 0}); continue
        m, bs = r["mcnemar"], r["bootstrap"]
        rows.append({"comparison": c["name"], "a": c["a"], "b": c["b"], "n": r["n"],
                     "acc_a": m["accuracy_a"], "acc_b": m["accuracy_b"],
                     "acc_diff_b_minus_a": bs["accuracy"]["diff_b_minus_a"],
                     "acc_ci95": (round(bs["accuracy"]["ci_low"], 3), round(bs["accuracy"]["ci_high"], 3)),
                     "macro_f1_diff": bs["macro_f1"]["diff_b_minus_a"],
                     "macro_f1_ci95": (round(bs["macro_f1"]["ci_low"], 3), round(bs["macro_f1"]["ci_high"], 3)),
                     "b_right_a_wrong": m["b_right_a_wrong"], "a_right_b_wrong": m["a_right_b_wrong"],
                     "mcnemar_p_exact": m["p_value_exact"], "odds_ratio": m["odds_ratio_b_over_a"],
                     "dropped_unpaired": r["pairing"]["dropped_unpaired"], "note": m["note"]})
    return pd.DataFrame(rows)

## Results on this run

In [ ]:
labelled = {p["claim_id"] for p in run["predictions"]} & set(gold)
if not labelled:
    banner("No finalized gold labels for this run's claims, so no test is computed on it. "
           "A significance test on unlabelled or mock data would be meaningless.")
else:
    res = report(run["predictions"], gold, comparisons)
    print("saved", save_table(res, "statistical_tests", run))
    if res.n.max() < 30:
        banner(f"Only {res.n.max()} paired claims: confidence intervals will be wide and the test has little power.")
    display(res)

## Demonstration on SYNTHETIC data (not research results)

Shows the full report on made-up labels (A3 drawn with 70% accuracy, A2 60%, SR 62%) so the output format and the code path are visible before real gold labels exist.

In [ ]:
from engine import synthetic
from engine.datasets import finalized_gold
banner("SYNTHETIC demonstration -- these numbers say nothing about any approach.", "info")
sc = synthetic.synthetic_claims(120)
sa, sj = synthetic.synthetic_annotations(sc)
sgold = finalized_gold(sc, sa, sj)
spreds = synthetic.synthetic_predictions(sgold, {"A2_rag_numeric": 0.60, "A3_contradiction_aware": 0.70, "SR_self_reflection": 0.62})
report(spreds, sgold, [{"name": "main", "a": "A2_rag_numeric", "b": "A3_contradiction_aware"},
                       {"name": "second_look_control", "a": "SR_self_reflection", "b": "A3_contradiction_aware"}], n_boot=2000)

## Assumptions and limitations

- **Pairing.** Both approaches are scored on the same claim ids; McNemar uses only discordant pairs (claims one approach got right and the other wrong). With fewer than ~25 discordant pairs the exact binomial p-value is the one to read.
- **Independence.** Claims are treated as independent draws. Several claims about the same company and period share filings, so the effective sample size is smaller than the claim count; report per-company results (notebook 05) alongside.
- **Bootstrap.** Percentile intervals from resampling claims with replacement (fixed seed); they describe uncertainty over this claim population, not over other companies, periods or models.
- **Multiple comparisons.** Two planned comparisons (main, second-look control) are pre-registered in the config; any further comparison should be reported as exploratory or corrected (e.g. Holm).
- **Effect size.** Report the accuracy / macro-F1 difference with its interval and the discordant-pair odds ratio, not only the p-value.
- **Scope.** A significant difference means the contradiction-aware pipeline did better on this claim set, with this model snapshot, corpus and prompt version. It is not evidence that the method is better in general.
- **Model determinism.** Temperature 0 reduces but does not remove run-to-run variation; cached responses make a run reproducible, and repeated live runs are needed to estimate that variation.